# 1.Model OI介绍

Model I/O 模块是与语言模型（LLMs）进行交互的核心组件，在整个框架中有着很重要的地位。
所谓的Model I/O，包括输入提示(Format)、调用模型(Predict)、输出解析(Parse)。分别对应着
Prompt Template ， Model 和Output Parser 。

简单来说，就是输⼊、模型处理、输出这三个步骤。

# 2.调用模型

## 2.1 模型的分类方式

角度1：按照模型的功能不同

 非对话模型（LLMs、Text Models）
 对话模型（Chat Models） （推荐）
 嵌入模型（Embedding Models）

角度2：模型调用时，几个参数的书写位置不同
 
 硬编码：写在代码中
 使用环境变量
 使用配置文件 （推荐）

角度3：具体调用的API
 
 OpenAI提供的API
 其他大模型自家提供的API
 LangChain的统一方式调用API （推荐）

## 2.2 角度1：按功能列举

类型1：非对话模型（LLMs、Text Models）
仅需单次文本生成任务（如摘要生成、翻译、代码生成、单次问答）或对接不支持消息
结构的旧模型（如部分本地部署模型）（ 言外之意，优先推荐ChatModel ）
不支持多轮对话上下文。每次调用独立处理输入，无法自动关联历史对话（需手动拼接历史文
本）。

In [ ]:
import os
import dotenv
from langchain_openai import OpenAI

dotenv.load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")

###########核心代码############
llm = OpenAI()

str = llm.invoke("thinkpad怎么调分辨率") # 直接输入字符串

print(str)

类型2：对话模型 ChatModel

聊天模型、对话模型，底层使用LLMs

输入：接收消息列表List[BaseMessage] 或PromptValue，每条消息需指定角色（如SystemMessage、HumanMessage、AIMessage）
输出：总是返回带角色的消息对象（BaseMessage子类），通常是AIMessage

原生支持多轮对话。通过消息列表维护上下文（例如： [SystemMessage, HumanMessage,AIMessage, ...] ），模型可基于完整对话历史生成回复。
适用场景：对话系统（如客服机器人、长期交互的AI助手）

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
import os
import dotenv

dotenv.load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")

########核心代码############
chat_model = ChatOpenAI(model="gpt-4o-mini")

messages = [
    SystemMessage(content="我是人工智能助手，我叫小智"),
    HumanMessage(content="你好，我是小明，很高兴认识你")
]

response = chat_model.invoke(messages) # 输入消息列表

print(type(response)) # <class 'langchain_core.messages.ai.AIMessage'>
print(response.content)

<class 'langchain_core.messages.ai.AIMessage'>
你好，小明！很高兴认识你！有什么我可以帮助你的吗？


类型3：Embedding模型

In [4]:
import os
import dotenv
from langchain_openai import OpenAIEmbeddings

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

#############
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-ada-002"
)

res1 = embeddings_model.embed_query('我是文档中的数据')
print(res1)
# 打印结果：[-0.004306625574827194, 0.003083756659179926, -0.013916781172156334, ...., ]


[-0.032839685678482056, 0.002757745562121272, -0.010489282198250294, -0.021119264885783195, -0.016841946169734, 0.02123182639479637, -0.03033520095050335, 0.0030461831483989954, -0.01864292286336422, -0.024594588205218315, 0.008948602713644505, 0.031038707122206688, -0.007893343456089497, 0.008610920049250126, -0.004913993179798126, 0.01269829273223877, 0.010165669023990631, -0.003032113192602992, 0.009342566132545471, -0.01716555841267109, -0.0017429373692721128, 0.010341545566916466, -0.0001418005267623812, -0.0003348251339048147, -0.01663089357316494, 0.01847408153116703, 0.013676166534423828, -0.022413717582821846, -0.001504624611698091, -0.01528016198426485, 0.02521367371082306, -0.0197403933852911, -0.020415758714079857, -0.018009766936302185, -0.0012698292266577482, -0.007991833612322807, -0.009645074605941772, -0.022990593686699867, 0.014815847389400005, -0.007717466447502375, 0.009652109816670418, 0.013113361783325672, -0.0027858857065439224, 0.018009766936302185, 0.0053220270

## 2.3 角度2：参数位置不同举例

模型调用的主要方法及参数：

 创建一个模型对象
 OpenAI(...)   非对话类
 ChatOpenAI(...)   对话类

 model.invoke(...)  执行调用，将用户输入发送给模型
 .content ：提取模型返回的实际文本内容

必须设置的参数
 base_url:大模型API服务的根目录
 api_key:用于身份验证的密钥，由大模型服务商（如 OpenAI、百度千帆）提供
 model/model_name:指定要调用的具体大模型名称（如 gpt-4-turbo 、ERNIE-3.5-8K 等）
 temperature ：温度，控制生成文本的“随机性”，取值范围为0～1

### 方式1：硬编码

#### 对话模型示例

In [5]:

from langchain_openai import ChatOpenAI
# 硬编码 API Key 和模型参数
chat_model = ChatOpenAI(
    api_key="sk-uUjocU4UYO8s9FBxX4zdYt5CnL0GufAh0Hyw1ErLBZ92qqSZ", # 明文暴露密钥
    base_url="https://api.openai-proxy.org/v1",
    model="gpt-3.5-turbo",
)
# 调用示例
response = chat_model.invoke("解释神经网络原理")
print(response.content)   #对话模型输出的是AIMessage 因此这里就需要.content,提取回复的文本内容

神经网络是由神经元组成的计算模型，模拟人类大脑的工作原理。神经网络由输入层、隐藏层和输出层组成，其中每一层由多个神经元组成，神经元之间通过连接来传递信息。

神经网络的工作原理是通过学习输入数据的模式，从而实现数据的分类、识别和预测等任务。当数据输入到神经网络中时，会经过一系列的运算和激活函数处理，最终输出结果。

神经网络的训练过程是通过反向传播算法来不断调整神经元之间的连接权重，使得网络的输出结果与实际结果更加接近。这样的训练过程可以提高神经网络的准确性和泛化能力。

总的来说，神经网络利用神经元的连接和权重来建立复杂的数据模型，通过训练来学习数据的模式，从而实现各种不同的任务。


#### 非对话模型示例

In [6]:

from langchain_openai import OpenAI
# 硬编码 API Key 和模型参数
llm = OpenAI(
    api_key="sk-uUjocU4UYO8s9FBxX4zdYt5CnL0GufAh0Hyw1ErLBZ92qqSZ", # 明文暴露密钥
    base_url="https://api.openai-proxy.org/v1",
    # model="gpt-3.5-turbo",   这里注释掉，用默认模型
)
# 调用示例
response = llm.invoke("解释神经网络原理")
print(response)   #非对话模型输出的是字符串str 因此这里就不需要.content

（5分）

1.神经网络是一种模拟人类大脑神经系统的计算模型，它由神经元和连接权重组成。神经元是神经网络的基本单元，它接收来自其他神经元的输入信号，并通过激活函数将加权输入信号转换为输出信号。神经元之间的连接权重决定了输入信号对神经元输出的影响程度。

2.神经网络通过多层神经元组成，每一层都有若干个神经元，层与层之间通过连接权重相连。第一层被称为输入层，用于接收外部输入信号；最后一层被称为输出层，用于输出网络的计算结果；中间的层被称为隐藏层，用于处理输入和输出之间的关系。

3.神经网络的训练过程就


### 方式2：环境变量配置

优点：密钥与代码分离，适合单机开发
缺点 ：切换终端或文件后，变量失效，需重新设置。

### 方式3：配置文件方式

使用 python-dotenv 加载本地配置文件，支持多环境管理（开发/生产）。

#### 方式1：

In [7]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
import os
load_dotenv() # 自动加载 .env 文件
print(os.getenv("OPENAI_API_KEY"))
print(os.getenv("OPENAI_BASE_URL"))
llm = ChatOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"), # 安全读取
    base_url=os.getenv("OPENAI_BASE_URL"),
    model="gpt-4o-mini",
    temperature=0.7,
)
response = llm.invoke("RAG 技术的核心流程")
print(response.content)

sk-uUjocU4UYO8s9FBxX4zdYt5CnL0GufAh0Hyw1ErLBZ92qqSZ
https://api.openai-proxy.org/v1
RAG（Retrieval-Augmented Generation）技术是一种结合信息检索和生成模型的自然语言处理方法，通常用于提高生成模型的回答质量和准确性。其核心流程可以分为以下几个步骤：

1. **查询生成**：首先，根据用户输入的问题或请求，生成一个查询。这一步骤可以利用生成模型（如GPT等）来理解用户的意图并生成相应的查询。

2. **信息检索**：使用生成的查询在一个外部知识库或文档数据库中检索相关信息。这可以是搜索引擎、知识图谱、数据库等。目标是找到与用户问题最相关的文档或信息片段。

3. **上下文整合**：将检索到的相关信息与用户的原始查询结合起来，形成一个包含上下文的输入。这通常涉及到将检索到的文本与用户问题进行融合，以便生成更准确的回答。

4. **生成回答**：利用生成模型（如基于Transformer的模型）根据整合后的上下文信息生成最终的回答。这一阶段模型会根据提供的上下文生成自然流畅的文本。

5. **结果展示**：将生成的答案返回给用户，可能还会包括检索到的信息来源，以增强回答的可靠性。

RAG技术的优势在于它能够结合外部知识，克服单一生成模型的知识局限性，提高回答的准确性和丰富性。


#### 方式2：给os内部的环境变量赋值

In [9]:
from langchain_openai import ChatOpenAI
import dotenv
dotenv.load_dotenv()
import os
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_API_BASE"] = os.getenv("OPENAI_BASE_URL")

text = "猫王是猫吗？"
chat_model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.7,
    max_tokens=300,
)
response = chat_model.invoke(text)
print(type(response))
print(response.content)

<class 'langchain_core.messages.ai.AIMessage'>
“猫王”是指美国著名的摇滚歌手埃尔维斯·普雷斯利（Elvis Presley），他的昵称“猫王”来源于他的舞台风格和对音乐的影响，并不是字面上的猫。因此，猫王并不是猫，而是一个人。


## 2.4角度3：各个平台的API调用

### OpenAI 官方API

In [ ]:
#调用非对话模型
from openai import OpenAI
# 从环境变量读取API密钥（推荐安全存储）
client = OpenAI(api_key="sk-uUjocU4UYO8s9FBxX4zdYt5CnL0GufAh0Hyw1ErLBZ92qqSZ", #填写自己的api-key
    base_url="https://api.openai-proxy.org/v1") #通过代码示例获取
# 调用Completion接口
response = client.completions.create(
    model="gpt-3.5-turbo-instruct", # 非对话模型
    prompt="请将以下英文翻译成中文：\n'Artificial intelligence will reshape the future.'",
    max_tokens=100, # 生成文本最大长度
    temperature=0.7, # 控制随机性
)
# 提取结果
print(response.choices[0].text.strip())

人工智能将重塑未来。


In [ ]:
#调用对话模型 写法1
from openai import OpenAI
client = OpenAI(api_key="sk-uUjocU4UYO8s9FBxX4zdYt5CnL0GufAh0Hyw1ErLBZ92qqSZ", #填写自己的api-key
    base_url="https://api.openai-proxy.org/v1")
completion = client.chat.completions.create(
    model="gpt-3.5-turbo", # 对话模型
    messages=[
        {"role": "system", "content": "你是一个乐于助人的智能AI小助手"},
        {"role": "user", "content": "你好，请你介绍一下你自己"}
    ],
    max_tokens=150,
    temperature=0.5
)
print(completion.choices[0].message)

ChatCompletionMessage(content='你好，我是一个智能AI助手，可以回答各种问题，提供帮助和建议。无论是日常生活中的问题、学习上的困惑，还是其他方面的需求，我都会尽力为您提供支持和帮助。有什么可以帮到您的吗？', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)


In [3]:
#调用对话模型 写法2
from openai import OpenAI
import os
import dotenv

dotenv.load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY3")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL3")


client = OpenAI()
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "system", "content": "我是一位乐于助人的AI智能小助手"},
        {"role": "user", "content": "你好，请你介绍一下你自己。"}
    ]
)
print(response.choices[0])

Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='你好，我是一位AI智能助手，专门设计来帮助回答问题、提供信息和解决问题。我可以回答有关历史、科学、技术、健康、教育和许多其他主题的问题。如果你有任何疑问，随时都可以问我哦！', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))


### 硅基流动API

In [10]:
from openai import OpenAI
import os
import dotenv
dotenv.load_dotenv()
client = OpenAI(
    api_key=os.getenv("SILICONFLOW_API_KEY"),
    base_url=os.getenv("SILICONFLOW_BASE_URL")
)

response = client.chat.completions.create(
    model="Pro/zai-org/GLM-4.7",
    messages=[
        {"role": "system", "content": "你是一个有用的助手"},
        {"role": "user", "content": "你好，请介绍一下你自己"}
    ],
    temperature=0.7,
    max_tokens=1000
)
print(response.choices[0].message.content)

你好！我是Z.ai训练的GLM大语言模型，我的设计目的是通过自然语言处理技术为用户提供信息和帮助。

我被训练用于理解问题、生成文本、回答知识和提供创意建议，不过我也有知识边界，无法进行实时互动或执行物理操作。我致力于以尊重、有益的方式与用户交流，同时保护用户隐私。

有什么我能帮助你解答的问题或提供协助的领域吗？


### 获取大模型的标准方式

In [ ]:
#非对话模型
import os
import dotenv
from langchain_openai import OpenAI

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY3")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL3")
llm = OpenAI( #非对话模型
    #max_tokens=512,
    #temperature=0.7,
)

In [ ]:
#对话模型
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY3")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL3")

chat_model = ChatOpenAI( #对话模型
    model="gpt-4o-mini",
    #max_tokens=512,
    #temperature=0.7,
    messages =[
        {"role":"system", "content": "我是一位乐于助人的AI智能小助手"},
        {"role": "user", "content": "你好，请你介绍一下你自己。"}
        ]
)

#配置文件见.env文件

#  3. 再谈Model IO调用模型

标准的对话模型调用过程：

invoke（）的输入可以是多种类型，典型的类型有：1）字符串类型；2）消息列表

invoke（）的输出类型BaseMessage的子类，AIMessage

In [4]:
from langchain_openai import ChatOpenAI
import os
import dotenv

# 加载配置文件
dotenv.load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_API_BASE"] = os.getenv("OPENAI_BASE_URL")


# 1.获取对话模型
chat_model = ChatOpenAI(
    # api_key=os.getenv("OPENAI_API_KEY"),
    # base_url=os.getenv("OPENAI_BASE_URL"),
    model="gpt-4o-mini",
)

# 2.调用对话模型
response = chat_model.invoke("RAG 技术的核心流程")

# 3.处理响应数据
print(response.content)

RAG（Retrieval-Augmented Generation）技术是一种结合了信息检索和文本生成的自然语言处理方法。它的核心流程通常包括以下几个步骤：

1. **查询生成**：首先，根据用户的输入或问题生成一个查询。这可能涉及对输入的理解和提取关键信息。

2. **信息检索**：使用生成的查询从一个外部知识库或文档集合中检索相关信息。这可以是搜索数据库、知识库或互联网上的文档。

3. **文档处理**：对检索到的文档进行处理，提取出有用的信息。这个步骤可能涉及自然语言处理技术，例如文本摘要、实体识别等。

4. **文本生成**：将用户的输入与检索到的信息结合起来，通过生成模型（例如 GPT 或 T5 等）生成最终的响应或答案。

5. **输出优化**：对生成的文本进行优化，确保其流畅性和相关性，有时还可能包括对生成内容的后处理，比如去除错误信息或调整语气。

6. **用户反馈**：某些系统会根据用户对生成结果的反馈进行调整和学习，从而不断改进其性能。

这种方法通过结合检索和生成的优势，在很多实际应用中都表现出了优越的性能，特别是在需要提供丰富信息或上下文的任务中。


## 3.1 关于对话模型的Message

LangChain的内置消息类型：

SystemMessage:设定AI行为规则或背景信息。比如设定AI的初始状态、行为模式或对话的总
体目标。比如“作为一个代码专家”，或者“返回json格式”。通常作为输入消息序列中的第一个
传递。
HumanMessage:表示来自用户输入。比如“实现 一个快速排序方法”
AIMessege:存储AI回复的内容。这可以是文本，也可以是调用工具的请求
ChatMessage:可以自定义角色的通用消息类型
FunctionMessage/ToolMessage:函数调用/工具消息，用于函数调用结果的消息类型


你是一个数据分析师，请帮我分析今年出生率情况
AI回复

In [5]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [SystemMessage(content="你是一位乐于助人的智能小助手"),
    HumanMessage(content="你好，请你介绍一下你自己"),]

print(messages)

response = chat_model.invoke(messages)
print(response.content)

[SystemMessage(content='你是一位乐于助人的智能小助手', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好，请你介绍一下你自己', additional_kwargs={}, response_metadata={})]
你好！我是一个智能助手，旨在提供信息、回答问题和解决各种问题。我可以帮助你获取知识、提供建议或进行日常任务的管理。无论你有什么问题，请随时问我！


In [2]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
messages = [
    SystemMessage(content=["你是一个数学家,只会回答数学问题","每次你都能给出详细的方案"]),
    HumanMessage(content="1 + 2 * 3 = ?"),
    AIMessage(content="1 + 2 * 3 的结果是7"),
]
print(messages)

[SystemMessage(content=['你是一个数学家,只会回答数学问题', '每次你都能给出详细的方案'], additional_kwargs={}, response_metadata={}), HumanMessage(content='1 + 2 * 3 = ?', additional_kwargs={}, response_metadata={}), AIMessage(content='1 + 2 * 3 的结果是7', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


In [6]:
#1.导入相关包
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
# 2.直接创建不同类型消息
systemMessage = SystemMessage(
    content="你是一个AI开发工程师",
    additional_kwargs={"tool": "invoke_tool()"}
)
humanMessage = HumanMessage(
    content="你能开发哪些AI应用?"
)
aiMessage = AIMessage(
    content="我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等"
)
# 3.打印消息列表
messages = [systemMessage,humanMessage,aiMessage]
print(messages)

[SystemMessage(content='你是一个AI开发工程师', additional_kwargs={'tool': 'invoke_tool()'}, response_metadata={}), HumanMessage(content='你能开发哪些AI应用?', additional_kwargs={}, response_metadata={}), AIMessage(content='我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


In [ ]:
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage,
    ChatMessage
)
# 创建不同类型的消息
system_message = SystemMessage(content="你是一个专业的数据科学家")
human_message = HumanMessage(content="解释一下随机森林算法")
ai_message = AIMessage(content="随机森林是一种集成学习方法...")#可以自己定义一个回复内容，很少会这样
custom_message = ChatMessage(role="analyst", content="补充一点关于超参数调优的信息")#使用的不多
print(system_message.content)
print(human_message.content)
print(ai_message.content)
print(custom_message.content)

你是一个专业的数据科学家
解释一下随机森林算法
随机森林是一种集成学习方法...
补充一点关于超参数调优的信息


In [10]:
# 结合大模型使用
import os
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

import dotenv
dotenv.load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY3")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL3")

chat_model = ChatOpenAI(model="gpt-4o-mini")


messages = [
    SystemMessage(content="你是一位擅长人工智能相关学科的专家"),
    HumanMessage(content="你好，请解释一下什么是机器学习？"),
]

response = chat_model.invoke(messages) # 输入消息列表
print(response.content) 
print(type(response))

你好！机器学习是一种人工智能的子领域，它使计算机系统能够通过经验自动改进其性能，而无需明确编程。换句话说，机器学习允许计算机根据数据进行模式识别和预测。

机器学习的核心概念可以归纳为以下几个方面：

1. **数据驱动**：机器学习依赖大量数据，算法通过分析这些数据寻找模式，从中学习。

2. **模型**：在学习过程中，机器学习算法会创建一个模型，该模型能够对新输入数据进行预测或分类。

3. **训练与测试**：机器学习过程通常分为训练和测试两个阶段。训练阶段用来调整模型参数，以便使模型能更好地拟合训练数据；测试阶段则用来评估模型在未知数据上的表现。

4. **算法**：机器学习包含多种算法，主要分为监督学习（使用带标签的数据进行训练）、无监督学习（使用无标签的数据进行训练）和强化学习（通过与环境互动不断改善决策）。

5. **应用领域**：机器学习广泛应用于多个领域，如图像识别、自然语言处理、推荐系统、金融预测、医疗诊断等。

总体而言，机器学习的目标是让计算机能够自动从数据中学习并作出决策，以处理各类复杂任务。
<class 'langchain_core.messages.ai.AIMessage'>


## 3.2 关于多轮对话与上下文记忆

In [3]:
# 获取大模型
import os
from langchain_openai import ChatOpenAI
import dotenv
dotenv.load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")   

chat_model = ChatOpenAI(model="gpt-4o-mini")

# 测试1   
from langchain_core.messages import SystemMessage, HumanMessage
sys_message = SystemMessage(
    content="我是一个人工智能的助手，我的名字叫小智",
)
human_message = HumanMessage(content="猫王是一只猫吗？")
messages = [sys_message, human_message]

#调用大模型，传入messages
response = chat_model.invoke(messages)

print(response.content)

response1 = chat_model.invoke("你叫什么名字？")#实际上结果中无法记住它叫小智
print(response1.content)

“猫王”通常指的是美国摇滚乐传奇艾尔维斯·普雷斯利（Elvis Presley），因为“猫王”是他的中文昵称。不过，如果你是指某个特定的“猫王”作为一只猫，那可能是指某个著名的猫咪或网络猫咪角色，需要更多的信息来确认。请问你指的是哪一个？
我是一个人工智能助手，没有具体的名字。你可以叫我“助手”或者“AI”。有什么我可以帮助你的吗？


In [ ]:
# 测试2
from langchain_core.messages import HumanMessage, SystemMessage

sys_message = SystemMessage(
    content="你是一位乐于助人的智能AI小助手，名字叫做小智",
)
human_message = HumanMessage(content="猫王是一只猫吗？")#这个信息大模型不会响应，AI只会回复最后一个问题
human_message1 = HumanMessage(content="你叫什么名字？")

messages = [sys_message, human_message, human_message1]

#调用大模型，传入messages
response = chat_model.invoke(messages)
print(response.content)

# sys_message和human_message一同传入模型，模型可以对sys_message和最后一个human_message做出回应

我叫小智，很高兴为你服务！有什么我可以帮助你的吗？


In [ ]:
# 测试3
from langchain_core.messages import HumanMessage, SystemMessage

sys_message = SystemMessage(
    content="我是一个人工智能的助手，是一位乐于助人的智能AI小助手，我的名字叫小智",
)

human_message = HumanMessage(content="猫王是一只猫吗？")#这个信息大模型会响应

sys_message1 = SystemMessage(
    content="我叫小A",
)

human_message1 = HumanMessage(content="你叫什么名字？") 
messages = [sys_message,human_message,sys_message1, human_message1]#消息是一次性传给模型的，因此模型可以记住小智
#调用大模型，传入messages
response = chat_model.invoke(messages)
print(response.content)

# sys_message和human_message一同传入模型，模型可以对第一个sys_message和最后一个human_message做出回应

我叫小智，很高兴认识你！你有什么问题或者需要帮助的吗？


In [8]:
# 测试4
from langchain_core.messages import HumanMessage, SystemMessage

# 第一组
sys_message = SystemMessage(
    content="我是一个人工智能的助手，是一位乐于助人的智能AI小助手，我的名字叫小智",
)
human_message = HumanMessage(content="猫王是一只猫吗？")

messages = [sys_message, human_message]

# 第二组
sys_message1 = SystemMessage(
    content="我可以做很多事，有需要就找我吧",
)
human_message1 = HumanMessage(content="你叫什么名字？")

messages1 = [sys_message1, human_message1]

#调用大模型，传入messages
response = chat_model.invoke(messages)
print(response.content)

response1 = chat_model.invoke(messages1)#小智在messages里面，因此模型不会记得小智
print(response1.content)


“猫王”通常是指美国著名摇滚歌手埃尔维斯·普雷斯利（Elvis Presley），因为“猫王”（King of Rock and Roll）这个称号而闻名。他实际上是一位人类，而不是一只猫。不过，在某些非正式或幽默的用法中，有时大家可能会用“猫王”来形容一只非常特别或者有趣的猫。你指的是哪种情况呢？
我是一个人工智能助手，没有具体的名字，你可以叫我助手或AI。如果你有任何问题或需要帮助，随时告诉我！


In [8]:
# 测试5
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

messages = [SystemMessage(content="我是一个人工智能助手，我的名字叫小智"),
    HumanMessage(content="人工智能英文怎么说？"),
    AIMessage(content="Artificial Intelligence"),
    HumanMessage(content="你叫什么名字？")
]

messages1 =[
    SystemMessage(content="我是一个人工智能助手，我的名字叫小智"),
    HumanMessage(content="很高兴认识你"),
    AIMessage(content="我也很高兴认识你"),
    HumanMessage(content="你叫什么名字？")
]

messages2 = [
    SystemMessage(content="我是一个人工智能助手，我的名字叫小智"),
    HumanMessage(content="人工智能英文怎么说？"),
    AIMessage(content="AI"),
    HumanMessage(content="你叫什么名字"),
]

chat_model.invoke(messages2)

AIMessage(content='我叫小智。你可以问我任何问题！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 40, 'total_tokens': 53, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f97eff32c5', 'id': 'chatcmpl-Czx0twjXvaZVQWERQmElUNtz11Rdg', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019bd984-8032-7be1-9a84-67881957d7cb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 40, 'output_tokens': 13, 'total_tokens': 53, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

在档次的会话中有记忆。

## 3.3 关于模型调用的方法

为了尽可能简化自定义链的创建，我们实现了一个"Runnable"协议。许多LangChain组件实现了
Runnable 协议，包括聊天模型、提示词模板、输出解析器、检索器、代理(智能体)等。

### 流式输出与非流式输出

非流式输出：invoke 这是Langchain与LLM交互时的默认行为，是最简单、最稳定的语言模型调用方式。
当用户发出请求后，系统在后台等待模型生成完整响应，然后一次性将全部结果返回。

流式输出：一种更具交互感的模型输出方式，用户不再需要等待完整答案，而是能看到模型逐个
token 地实时返回内容。

In [14]:
# 非流式输出
import os
import dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
dotenv.load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")   

chat_model = ChatOpenAI(model="gpt-4o-mini")
# 支持多个消息作为输入
messages = [
SystemMessage(content="你是一位乐于助人的助手。你叫于老师"),
HumanMessage(content="你是谁？")
]
response = chat_model.invoke(messages)
print(response.content)

你好，我是于老师，很高兴能为你提供帮助！有什么问题或者需要我协助的事情吗？


In [8]:
# 流式输出
import os
import dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
dotenv.load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")   

chat_model = ChatOpenAI(model="gpt-4o-mini", 
                        streaming=True#开启流式输出
                        )
# 支持多个消息作为输入
messages = [
HumanMessage(content="你是谁？请介绍一下自己")
]

# 流式调用LLM获取响应
print("开始流式输出：")
for chunk in chat_model.stream(messages):
# 逐个打印内容块
    print(chunk.content, end="", flush=True) # 刷新缓冲区 (无换行符，缓冲区未刷新，内容可能不会立即显示)
print("\n流式输出结束")

开始流式输出：
我是一个人工智能助手，旨在回答问题、提供信息和协助解决各种问题。我可以帮助您获取知识、提供建议或满足其他需求。有什么我可以帮您的吗？
流式输出结束


### 批量调用

In [6]:
import os
import dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
dotenv.load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
# 初始化大模型
chat_model = ChatOpenAI(model="gpt-4o-mini")
messages1 = [SystemMessage(content="你是一位乐于助人的智能小助手"),
    HumanMessage(content="请帮我介绍一下什么是机器学习"), ]
messages2 = [SystemMessage(content="你是一位乐于助人的智能小助手"),
    HumanMessage(content="请帮我介绍一下什么是AIGC"), ]
messages3 = [SystemMessage(content="你是一位乐于助人的智能小助手"),
    HumanMessage(content="请帮我介绍一下什么是大模型技术"), ]
messages = [messages1, messages2, messages3]
# 调用batch
response = chat_model.batch(messages)
print(response[0].content)
print(response[1].content)
print(response[2].content)

机器学习是人工智能（AI）的一个子领域，涉及让计算机系统通过经验（数据）自动改进其性能的过程，而无需明确编程。简单来说，机器学习使计算机能够从数据中学习并做出预测或决策。

机器学习的基本流程通常包括以下几个步骤：

1. **数据收集**：获取相关的训练数据，这些数据用于模型的学习和训练。

2. **数据预处理**：对原始数据进行清洗、转化和整理，以适应模型要求。

3. **选择算法**：根据问题类型选择合适的机器学习算法。常见算法包括决策树、支持向量机、神经网络等。

4. **训练模型**：使用训练数据来训练模型，从而使其能够识别数据中的模式。

5. **模型评估**：使用测试数据评估模型的性能，确保其准确性和可靠性。

6. **模型优化**：根据评估结果调整模型参数，提高模型性能。

7. **部署和监控**：将训练好的模型应用于实际场景，并持续监控其表现，以便进行必要的调整。

机器学习主要分为以下几类：

- **监督学习**：模型在已标记的数据上进行训练，目的是学习输入与输出之间的映射关系。常见应用有分类问题（如垃圾邮件检测）和回归问题（如房价预测）。

- **无监督学习**：在没有标记的数据上进行训练，目的是发现数据中的潜在结构或模式，例如聚类和降维。

- **半监督学习**：结合了监督学习和无监督学习，使用少量标记数据和大量未标记数据进行训练。

- **强化学习**：通过奖励和惩罚机制，让模型在环境中学习最优策略。常见应用包括游戏 AI 和机器人控制。

机器学习在各个领域都有广泛的应用，包括自然语言处理、图像识别、推荐系统、金融分析等。随着计算能力的提升和大数据的发展，机器学习的应用前景越来越广阔。
AIGC（Artificial Intelligence Generated Content）是指利用人工智能技术生成的内容。这种内容可以包括文本、图像、音频、视频等多种形式。AIGC的出现和发展得益于深度学习、自然语言处理和计算机视觉等领域的进步，使得机器能够理解并生成符合人类表现形式的作品。

AIGC的应用范围非常广泛，包括但不限于：

1. **文本生成**：例如，自动撰写新闻报道、博客文章、小说，或提供智能客服对话等。
2. **图像生成**：使用生成对抗网络（GAN）创建艺术作品、设计图、广告创意等。
3. **音频生成**

### 同步调用与异步调用(了解)

In [17]:
# 同步调用
import time
def call_model():
# 模拟同步API调用
    print("开始调用模型...")
    time.sleep(5) # 模拟调用等待,单位：秒
print("模型调用完成。")
def perform_other_tasks():
# 模拟执行其他任务
    for i in range(5):
        print(f"执行其他任务 {i + 1}")
        time.sleep(1) # 单位：秒
def main():
    start_time = time.time()
    call_model()
    perform_other_tasks()
    end_time = time.time()
    total_time = end_time - start_time
    return f"总共耗时：{total_time}秒"
# 运行同步任务并打印完成时间
main_time = main()
print(main_time)

模型调用完成。
开始调用模型...
执行其他任务 1
执行其他任务 2
执行其他任务 3
执行其他任务 4
执行其他任务 5
总共耗时：10.04110336303711秒


之前的llm.invoke(...) 本质上是一个同步调用。每个操作依次执行，直到当前操作完成后才开始下一个
操作，从而导致总的执行时间是各个操作时间的总和。

异步调用
异步调用，允许程序在等待某些操作完成时继续执行其他任务，而不是阻塞等待。这在处理I/O操作（如
网络请求、文件读写等）时特别有用，可以显著提高程序的效率和响应性。

In [18]:
# 写法1：此写法适合Jupyter Notebook
import asyncio
import time
async def async_call(llm):
    await asyncio.sleep(5) # 模拟异步操作
    print("异步调用完成")

async def perform_other_tasks():
    await asyncio.sleep(5) # 模拟异步操作
    print("其他任务完成")

async def run_async_tasks():
    start_time = time.time()
    await asyncio.gather(
        async_call(None), # 示例调用，使用None模拟LLM对象
        perform_other_tasks()
    )
    end_time = time.time()
    return f"总共耗时：{end_time - start_time}秒"
# # 正确运行异步任务的方式
# if __name__ == "__main__":
# # 使用 asyncio.run() 来启动异步程序
# result = asyncio.run(run_async_tasks())
# print(result)

# 在 Jupyter 单元格中直接调用
result = await run_async_tasks()
print(result)

异步调用完成
其他任务完成
总共耗时：5.004839181900024秒


使用asyncio.gather() 并行执行时，理想情况下，因为两个任务几乎同时开始，它们的执行时间将重
叠。如果两个任务的执行时间相同（这里都是5秒），那么总执行时间应该接近单个任务的执行时间，而
不是两者时间之和。

In [19]:
# 异步调用之ainvoke
# 举例1：验证ainvoke是否是异步

# 方式1
import inspect
print("ainvoke 是协程函数:", inspect.iscoroutinefunction(chat_model.ainvoke))
print("invoke 是协程函数:", inspect.iscoroutinefunction(chat_model.invoke))

ainvoke 是协程函数: True
invoke 是协程函数: False


# 4. Model I/O之Prompt Template

## 4.1 介绍与分类

Prompt Template，通过模板管理大模型的输入。

Prompt Template 是LangChain中的一个概念，接收用户输入，返回一个传递给LLM的信息（即提示词prompt）。
在应用开发中，固定的提示词限制了模型的灵活性和适用范围。所以，prompt template 是一个模板化的字符串，你可以将变量插入到模板中，从而创建出不同的提示。调用时：
以字典作为输入，其中每个键代表要填充的提示模板中的变量。
输出一个PromptValue 。这个 PromptValue 可以传递给 LLM 或 ChatModel，并且还可以转换为字符串或消息列表。


有几种不同类型的提示模板：
PromptTemplate ：LLM提示模板，用于生成字符串提示。它使用 Python 的字符串来模板提示。
ChatPromptTemplate ：聊天提示模板，用于组合各种角色的消息模板，传入聊天模型。
XxxMessagePromptTemplate ：消息模板词模板，包括：SystemMessagePromptTemplate、
HumanMessagePromptTemplate、AIMessagePromptTemplate、
ChatMessagePromptTemplate等
FewShotPromptTemplate ：样本提示词模板，通过示例来教模型如何回答
PipelinePrompt ：管道提示词模板，用于把几个提示词组合在一起使用。
自定义模板：允许基于其它模板类来定制自己的提示词模板。

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import FewShotPromptTemplate
from langchain_core.prompts import (
ChatMessagePromptTemplate,
SystemMessagePromptTemplate,
AIMessagePromptTemplate,
HumanMessagePromptTemplate,
)

## 4.2 复习：str.format()

Python的str.format() 方法是一种字符串格式化的手段，允许在字符串中插入变量。使用这种方法，可以创建包含占位符的字符串模板，占位符由花括号{} 标识。
调用format()方法时，可以传入一个或多个参数，这些参数将被顺序替换进占位符中。
str.format()提供了灵活的方式来构造字符串，支持多种格式化选项。

在LangChain的默认设置下， PromptTemplate 使用 Python 的str.format() 方法进行模板化。这样
在模型接收输入前，可以根据需要对数据进行预处理和结构化。


带有位置参数的用法

In [15]:
# 使用位置参数
info = "Name: {0}, Age: {1}".format("Jerry", 25)
print(info)

Name: Jerry, Age: 25


带有关键字参数的用法

In [16]:
# 使用关键字参数
info = "Name: {name}, Age: {age}".format(name="Tom", age=25)
print(info)

Name: Tom, Age: 25


使用字典解包的方式

In [17]:
# 使用字典解包
person = {"name": "David", "age": 40}
info = "Name: {name}, Age: {age}".format(**person)
print(info)

Name: David, Age: 40


## 4.3 具体使用：PromptTemplate

### 4.3.1 使用说明

PromptTemplate类，用于快速构建包含变量的提示词模板，并通过传入不同的参数值生成自定义的提示词。

1、PromptTemplate如何获取实例（掌握两种方式： .format()   .from_template() ）

2、两种特殊结构的使用（部分提示词模板的使用、组合提示词的使用）

3、给变量赋值的两种方式(掌握  format（）  invoke（） )

4、结合大模型的使用

主要参数介绍：

template：定义提示词模板的字符串，其中包含文本和变量占位符（如{name}） ；

input_variables： 列表，指定了模板中使用的变量名称，在调用模板时被替换；

partial_variables：字典，用于定义模板中一些固定的变量名。这些值不需要再每次调用时被替换。

函数介绍：

format()：给input_variables变量赋值，并返回提示词。利用format() 进行格式化时就一定要赋
值，否则会报错。当在template中未设置input_variables，则会自动忽略。

### 4.3.2 两种实例化方式

方式1：使用构造方法

举例1：

In [19]:
from langchain_core.prompts import PromptTemplate
# 定义模板：描述主题的应用
template = PromptTemplate(template="请简要描述{topic}的应用。",
                          input_variables=["topic"])
print(template)

# 使用模板生成提示词
prompt_1 = template.format(topic="机器学习")
prompt_2 = template.format(topic="自然语言处理")

print("提示词1:", prompt_1)
print("提示词2:", prompt_2)

input_variables=['topic'] input_types={} partial_variables={} template='请简要描述{topic}的应用。'
提示词1: 请简要描述机器学习的应用。
提示词2: 请简要描述自然语言处理的应用。


可以直观的看到PromptTemplate可以将template中声明的变量topic准确提取出来，使prompt更清晰。

举例2：定义多变量模板

In [21]:
from langchain_core.prompts import PromptTemplate
#定义多变量模板
template = PromptTemplate(
    template="请评价{product}的优缺点，包括{aspect1}和{aspect2}。",
    input_variables=["product", "aspect1", "aspect2"])
#使用模板生成提示词
prompt_1 = template.format(product="智能手机", aspect1="电池续航", aspect2="拍照质量")
prompt_2 = template.format(product="笔记本电脑", aspect1="处理速度", aspect2="便携性")

print("提示词1:",prompt_1)
print("提示词2:",prompt_2)

提示词1: 请评价智能手机的优缺点，包括电池续航和拍照质量。
提示词2: 请评价笔记本电脑的优缺点，包括处理速度和便携性。


方式2：调用from_template() 推荐！！！

举例1：

In [22]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template(
    "请给我一个关于{topic}的{type}解释。"
)

#传入模板中的变量名
prompt = prompt_template.format(type="详细", topic="量子力学")

print(prompt)

请给我一个关于量子力学的详细解释。


举例2：模板支持任意数量的变量，包括不含变量：

In [23]:
#1.导入相关的包
from langchain_core.prompts import PromptTemplate
# 2.定义提示词模版对象
text = """
Tell me a joke
"""

prompt_template = PromptTemplate.from_template(text)

# 3.默认使用f-string进行格式化（返回格式好的字符串）
prompt = prompt_template.format()
print(prompt)


Tell me a joke



### 4.3.3 两种新的结构形式

形式1：部分提示词模版

在生成prompt前就已经提前初始化部分的提示词，实际进一步导入模版的时候只导入除已初始化的变量即可。


举例1：

方式1：实例化过程中使用partial_variables变量

In [ ]:
from langchain_core.prompts import PromptTemplate
#方式2：
template2 = PromptTemplate(
    template="{foo}{bar}",
    input_variables=["foo","bar"],
    partial_variables={"foo": "hello"}#将部分变量初始化
)
prompt2 = template2.format(bar="world")

print(prompt2)

helloworld


方式2：使用 PromptTemplate.partial() 方法创建部分提示模板

In [ ]:
from langchain_core.prompts import PromptTemplate
template1 = PromptTemplate(
    template="{foo}{bar}",
    input_variables=["foo", "bar"]
)

#方式1：
partial_template1 = template1.partial(foo="hello") #.partial()方法必须返回值，它不会改变'template1'本身
prompt1 = partial_template1.format(bar="world")
print(prompt1)

helloworld


举例2：直接在实例后面.partial()

In [ ]:
from langchain_core.prompts import PromptTemplate
# 完整模板
full_template = """你是一个{role}，请用{style}风格回答：
问题：{question}
答案："""

# 预填充角色和风格
partial_template = PromptTemplate.from_template(full_template).partial(
    role="资深厨师",
    style="专业但幽默"
)     #直接在实例后面用.partial()方法预填充变量

# 只需提供剩余变量
print(partial_template.format(question="如何煎牛排？"))

你是一个资深厨师，请用专业但幽默风格回答：
问题：如何煎牛排？
答案：


举例3：

In [28]:
prompt_template = PromptTemplate.from_template(
    template = "请评价{product}的优缺点，包括{aspect1}和{aspect2}。",
    partial_variables= {"aspect1":"电池","aspect2":"屏幕"}
)

prompt= prompt_template.format(product="笔记本电脑")
print(prompt)

请评价笔记本电脑的优缺点，包括电池和屏幕。


形式2：组合提示词(了解)

举例：

In [1]:
from langchain_core.prompts import PromptTemplate
template = (
    PromptTemplate.from_template("Tell me a joke about {topic}")
    + ", make it funny"
    + "\n\nand in {language}"
)

prompt = template.format(topic="sports", language="spanish")
print(prompt)

Tell me a joke about sports, make it funny

and in spanish


### 4.3.4 给变量赋值的两种方式：format() 与 invoke()

只要对象是RunnableSerializable接口类型，都可以使用invoke()，替换前面使用format()的调用方式。

format()，返回值为字符串类型；

invoke()，返回值为PromptValue类型，接着调用to_string()返回字符串。


举例1：调用invoke（）参数是字典

In [5]:
from langchain_core.prompts import PromptTemplate
#定义多变量模板
template = PromptTemplate(
    template="请评价{product}的优缺点，包括{aspect1}和{aspect2}。",
    input_variables=["product", "aspect1", "aspect2"])
#使用模板生成提示词
#prompt_1 = template.format(product="智能手机", aspect1="电池续航", aspect2="拍照质量")
prompt_1 = template.invoke({"product":"智能手机", "aspect1":"电池续航", "aspect2":"拍照质量"})
print(prompt_1)
print(type(prompt_1))

text='请评价智能手机的优缺点，包括电池续航和拍照质量。'
<class 'langchain_core.prompt_values.StringPromptValue'>


In [2]:
#1.导入相关的包
from langchain_core.prompts import PromptTemplate

# 2.定义提示词模版对象
prompt_template = PromptTemplate.from_template(
    "Tell me a {adjective} joke about {content}."
)

# 3.默认使用f-string进行格式化（返回格式好的字符串）
prompt_template.invoke({"adjective":"funny", "content":"chickens"})

StringPromptValue(text='Tell me a funny joke about chickens.')

举例2：

In [3]:
#1.导入相关的包
from langchain_core.prompts import PromptTemplate

# 2.使用初始化器进行实例化
prompt = PromptTemplate(
    input_variables=["adjective", "content"],
    template="Tell me a {adjective} joke about {content}")

# 3. PromptTemplate底层是RunnableSerializable接口 所以可以直接使用invoke()调用
prompt.invoke({"adjective": "funny", "content": "chickens"})

StringPromptValue(text='Tell me a funny joke about chickens')

举例3：

In [32]:
from langchain_core.prompts import PromptTemplate

prompt_template = (
    PromptTemplate.from_template("Tell me a joke about {topic}")
    + ", make it funny"
    + " and in {language}"
)

prompt = prompt_template.invoke({"topic":"sports", "language":"spanish"})
print(prompt)

text='Tell me a joke about sports, make it funny and in spanish'


### 4.3.5 结合LLM调用

Prompt 与大模型结合。

问题：这里的大模型，是哪类呢？非对话大模型？对话大模型？

提供大模型：（非对话大模型）

In [5]:
import os
import dotenv
from langchain_openai import OpenAI

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

llm = OpenAI()

提供大模型：（对话大模型）

In [3]:
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

llm1 = ChatOpenAI(
    model="gpt-4o-mini"
)

调用过程：

In [9]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template(
    template = "请评价{product}的优缺点，包括{aspect1}和{aspect2}。"
)

#prompt = prompt_template.format(product="电脑",aspect1="性能",aspect2="电池")
prompt = prompt_template.invoke({"product":"电脑","aspect1":"性能","aspect2":"电池"})
print(type(prompt))
#llm.invoke(prompt)  #使用非对话模型调用
llm1.invoke(prompt).content  #使用对话模型调用

<class 'langchain_core.prompt_values.StringPromptValue'>


'评价电脑的优缺点可以从多个角度进行分析，包括性能、电池、设计、价格等。以下是对电脑优缺点的综合评价：\n\n### 优点：\n\n1. **性能**：\n   - **处理能力强**：现代电脑通常配备高性能的处理器（如Intel i5/i7或AMD Ryzen系列），能够高效完成复杂任务。\n   - **多任务处理**：大多数电脑支持多任务处理，可以同时运行多个应用程序而不影响性能。\n   - **图形性能**：高性能的显卡（如NVIDIA或AMD）可支持游戏、图形设计和视频编辑等高负荷工作。\n\n2. **电池续航**：\n   - **长续航**：许多笔记本电脑具备良好的电池续航能力，可以提供6-12小时的使用时间，适合外出办公或旅行。\n   - **快速充电**：一些型号支持快速充电功能，能够在短时间内充入大量电量，提高便携性。\n\n3. **扩展性和升级性**：\n   - 很多电脑允许用户更换和升级内存、存储等部件，以满足日益增长的性能需求。\n\n4. **多样化选择**：\n   - 市场上有各种品牌和型号，用户可以根据具体需求选择适合的电脑。\n\n### 缺点：\n\n1. **性能**：\n   - **价格高**：高性能电脑通常价格较贵，对于预算有限的用户可能是个问题。\n   - **发热与噪音**：高性能电脑在高负荷工作时可能会发热，风扇噪音相对较大。\n\n2. **电池**：\n   - **电池寿命有限**：随着使用时间的增加，电池性能可能会下降，导致续航缩短。\n   - **固定电池设计**：有些型号的电池不可拆卸，用户在更换电池时需要专业服务。\n\n3. **便携性**：\n   - **重量**：高性能的游戏本或工作站通常较重，携带不便。\n   - **尺寸**：某些电脑可能尺寸较大，不易携带，尤其是在需要频繁移动的情况下。\n\n4. **系统兼容性**：\n   - 不同操作系统可能对软件支持程度不同，用户需要根据自己的需求选择合适的系统。\n\n### 总结：\n选择电脑时，需要根据自己的使用需求和预算进行权衡。高性能电脑虽然有其优势，但在电池续航、便携性和价格方面可能存在一些不足。了解这些优缺点后，可以更好地做出适合自己的选择。'

## 4.4 具体使用：ChatPromptTemplate

1、实例化的方式（两种方式：使用构造方法，from_messages()

2、调用提示词模板的几种方法：invoke()\format()\format_messages()\format_prompt()

3、更丰富的实例化参数类型

4、结合LLM

5、插入消息列表：MessagePlaceholder

### 4.4.1 使用说明


ChatPromptTemplate是创建聊天消息列表的提示模板。它比普通 PromptTemplate 更适合处理多角色、多轮次的对话场景。

特点：
支持 System / Human / AI 等不同角色的消息模板
对话历史维护

参数类型：列表参数格式是tuple类型（ role :str content :str 组合最常用）

元组的格式为：
(role: str | type, content: str | list[dict] | list[object])
其中 role 是：字符串（如 "system" 、"human" 、"ai" ）

### 4.4.2 两种实例化方式

方式1：使用构造方法

举例：

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
#创建实例
chat_prompt_template = ChatPromptTemplate(
    messages=[
        ("system", "你是一个AI助手. 你的名字是 {name}."),
        ("human", "我的问题是:{question}"),
    ],
    #input_variables=["name", "question"]  # 可选参数，自动从消息中提取
)

chat_prompt = chat_prompt_template.invoke(
    input={"name":"小谷AI","question":"如何学习人工智能?"}
)
print(chat_prompt)

messages=[SystemMessage(content='你是一个AI助手. 你的名字是 小谷AI.', additional_kwargs={}, response_metadata={}), HumanMessage(content='我的问题是:如何学习人工智能?', additional_kwargs={}, response_metadata={})]


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
#参数类型这里使用的是tuple构成的list
prompt_template = ChatPromptTemplate([
    # 字符串 role + 字符串 content
    ("system", "你是一个AI开发工程师. 你的名字是 {name}."),
    ("human", "你能开发哪些AI应用?"),
    ("ai", "我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等."),
    ("human", "{user_input}")
])
#调用invoke()方法，返回字符串
prompt = prompt_template.invoke(input={"name":"小谷AI","user_input":"你能帮我做什么?"})
print(type(prompt))
print(prompt)

<class 'langchain_core.prompt_values.ChatPromptValue'>
messages=[SystemMessage(content='你是一个AI开发工程师. 你的名字是 小谷AI.', additional_kwargs={}, response_metadata={}), HumanMessage(content='你能开发哪些AI应用?', additional_kwargs={}, response_metadata={}), AIMessage(content='我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='你能帮我做什么?', additional_kwargs={}, response_metadata={})]


方式2：调用from_messages()

举例1：

In [18]:
from langchain_core.prompts import ChatPromptTemplate

#创建实例
chat_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个AI助手. 你的名字是 {name}."),
    ("human", "我的问题是:{question}"),
])

response = chat_prompt_template.invoke(
    {"name":"小谷AI","question":"如何学习人工智能?"}
)
print(response)
print(type(response))
print(len(response.messages)) #包含两条消息


messages=[SystemMessage(content='你是一个AI助手. 你的名字是 小谷AI.', additional_kwargs={}, response_metadata={}), HumanMessage(content='我的问题是:如何学习人工智能?', additional_kwargs={}, response_metadata={})]
<class 'langchain_core.prompt_values.ChatPromptValue'>
2


### 4.2.3 调用提示词模板的几种方法

invoke()\format()\format_messages()\format_prompt()

方法1：invoke()  传入的字典，返回ChatPromptValue  给大模型的

In [20]:
from langchain_core.prompts import ChatPromptTemplate

#创建实例
chat_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个AI助手. 你的名字是 {name}."),
    ("human", "我的问题是:{question}"),
])

response = chat_prompt_template.invoke(
    {"name":"小谷AI","question":"如何学习人工智能?"}  #invoke()方法传入字典
)
print(response)
print(type(response))
print(len(response.messages)) #包含两条消息


messages=[SystemMessage(content='你是一个AI助手. 你的名字是 小谷AI.', additional_kwargs={}, response_metadata={}), HumanMessage(content='我的问题是:如何学习人工智能?', additional_kwargs={}, response_metadata={})]
<class 'langchain_core.prompt_values.ChatPromptValue'>
2


方法2：format() 传入变量的值，返回str

In [21]:
from langchain_core.prompts import ChatPromptTemplate

#创建实例
chat_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个AI助手. 你的名字是 {name}."),
    ("human", "我的问题是:{question}"),
])

response = chat_prompt_template.format(name="小谷AI",question="如何学习人工智能?")  #format()方法传入字典
print(response)
print(type(response))


System: 你是一个AI助手. 你的名字是 小谷AI.
Human: 我的问题是:如何学习人工智能?
<class 'str'>


方法3：format_messages()  传入变量的值，返回消息构成的list

In [22]:
from langchain_core.prompts import ChatPromptTemplate

#创建实例
chat_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个AI助手. 你的名字是 {name}."),
    ("human", "我的问题是:{question}"),
])

response = chat_prompt_template.format_messages(name="小谷AI",question="如何学习人工智能?")  #format()方法传入字典
print(response)
print(type(response))


[SystemMessage(content='你是一个AI助手. 你的名字是 小谷AI.', additional_kwargs={}, response_metadata={}), HumanMessage(content='我的问题是:如何学习人工智能?', additional_kwargs={}, response_metadata={})]
<class 'list'>


方法4：format_prompt() 传入变量的值，返回ChatPromptValue

In [23]:
from langchain_core.prompts import ChatPromptTemplate

#创建实例
chat_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个AI助手. 你的名字是 {name}."),
    ("human", "我的问题是:{question}"),
])

response = chat_prompt_template.format_prompt(name="小谷AI",question="如何学习人工智能?")  #format()方法传入字典
print(response)
print(type(response))


messages=[SystemMessage(content='你是一个AI助手. 你的名字是 小谷AI.', additional_kwargs={}, response_metadata={}), HumanMessage(content='我的问题是:如何学习人工智能?', additional_kwargs={}, response_metadata={})]
<class 'langchain_core.prompt_values.ChatPromptValue'>


总结：调用提示词模板的几种方法
方法1：invoke()           传入的字典{}，返回ChatPromptValue
方法2：format()           传入变量的值，返回str
方法3：format_messages()  传入变量的值，返回消息构成的list
方法4：format_prompt()    传入变量的值，返回ChatPromptValue

如何实现ChatPromptValue与list[messages]、字符串之间转换

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

#创建实例
chat_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个AI助手. 你的名字是 {name}."),
    ("human", "我的问题是:{question}"),
])
response = chat_prompt_template.invoke(
    {"name": "小谷AI", "question": "如何学习人工智能?"})  #invoke()方法传入字典
# response = chat_prompt_template.format_prompt(
# name="小谷AI",question="如何学习人工智能?")  #format_prompt()方法传入变量值
print(response)
print(type(response))

#将ChatPromptValue转换为消息列表
response_messages = response.to_messages()
print(response_messages)
print(type(response_messages))

#将ChatPromptValue转换为字符串
response_to_string = response.to_string()
print(response_to_string)
print(type(response_to_string))

messages=[SystemMessage(content='你是一个AI助手. 你的名字是 小谷AI.', additional_kwargs={}, response_metadata={}), HumanMessage(content='我的问题是:如何学习人工智能?', additional_kwargs={}, response_metadata={})]
<class 'langchain_core.prompt_values.ChatPromptValue'>
[SystemMessage(content='你是一个AI助手. 你的名字是 小谷AI.', additional_kwargs={}, response_metadata={}), HumanMessage(content='我的问题是:如何学习人工智能?', additional_kwargs={}, response_metadata={})]
<class 'list'>
System: 你是一个AI助手. 你的名字是 小谷AI.
Human: 我的问题是:如何学习人工智能?
<class 'str'>


### 4.2.4 更丰富的实例化参数类型

本质：不管用构造方法还是from_message（）方式创建ChatPromptTemplate的实例，本质上来讲。传入的都是消息构成的列表。

从调用上来讲，不管使用构造方法还是from_messages(), messages参数类型都是列表，但是列表元素的类型是多样的，可以是字符串类型、字典类型、消息类型、元组构成的列表（最常用，最简单，最基础）、Chat提示词模板类型、消息提示词模板类型

举例1：元组列表

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

#创建实例

#第一种方式 构造
chat_prompt_template = ChatPromptTemplate(
    messages=[
        ("system", "你是一个AI助手. 你的名字是 {name}."),
        ("human", "我的问题是:{question}"),
    ],
    #input_variables=["name", "question"]  # 可选参数，自动从消息中提取
)

#第二种方式 from_messages
chat_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个AI助手. 你的名字是 {name}."),
    ("human", "我的问题是:{question}"),
])

response = chat_prompt_template.invoke(
    {"name": "小谷AI", "question": "如何学习人工智能?"})  #invoke()方法传入字典
# response = chat_prompt_template.format_prompt(
# name="小谷AI",question="如何学习人工智能?")  #format_prompt()方法传入变量值
print(response)
print(type(response))

举例2 字符串

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

#创建实例


chat_prompt_template = ChatPromptTemplate.from_messages([
    "我的问题是{question}"   #默认的角色是human
])

response = chat_prompt_template.invoke(
    {"question": "如何学习人工智能?"})  #invoke()方法传入字典
# response = chat_prompt_template.format_prompt(
# name="小谷AI",question="如何学习人工智能?")  #format_prompt()方法传入变量值
print(response)
print(type(response))

messages=[HumanMessage(content='我的问题是如何学习人工智能?', additional_kwargs={}, response_metadata={})]
<class 'langchain_core.prompt_values.ChatPromptValue'>


举例3 字典类型

In [38]:
from langchain_core.prompts import ChatPromptTemplate

#创建实例


chat_prompt_template = ChatPromptTemplate.from_messages([
    {"role":"system", "content":"你是一个AI助手，你的名字是： {name}."},
    {"role":"human", "content":"我的问题是： {question}"}
])

response = chat_prompt_template.invoke(
    {"name": "小智", "question": "如何学习人工智能?"})  #invoke()方法传入字典
# response = chat_prompt_template.format_prompt(
# name="小谷AI",question="如何学习人工智能?")  #format_prompt()方法传入变量值
print(response.to_messages())
print(response.to_string())
print(type(response))

[SystemMessage(content='你是一个AI助手，你的名字是： 小智.', additional_kwargs={}, response_metadata={}), HumanMessage(content='我的问题是： 如何学习人工智能?', additional_kwargs={}, response_metadata={})]
System: 你是一个AI助手，你的名字是： 小智.
Human: 我的问题是： 如何学习人工智能?
<class 'langchain_core.prompt_values.ChatPromptValue'>


举例4 消息类型

In [40]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage
#创建实例


chat_prompt_template = ChatPromptTemplate.from_messages([
    SystemMessage(content="你是一个AI助手，你的名字是： {name}."),
    HumanMessage(content="我的问题是： {question}")
])

response = chat_prompt_template.invoke(
    {"name": "小智", "question": "如何学习人工智能?"})  #invoke()方法传入字典
# response = chat_prompt_template.format_prompt(
# name="小谷AI",question="如何学习人工智能?")  #format_prompt()方法传入变量值
print(response.to_messages())
print(response.to_string())
print(type(response))

[SystemMessage(content='你是一个AI助手，你的名字是： {name}.', additional_kwargs={}, response_metadata={}), HumanMessage(content='我的问题是： {question}', additional_kwargs={}, response_metadata={})]
System: 你是一个AI助手，你的名字是： {name}.
Human: 我的问题是： {question}
<class 'langchain_core.prompt_values.ChatPromptValue'>


举例5 Chat提示词模板类型

In [46]:
from langchain_core.prompts import ChatPromptTemplate

#使用BaseChatPromptTemplate（嵌套的ChatPromptTemplate）
nested_prompt_template1 = ChatPromptTemplate.from_messages([
    ("system", "你是一个AI助手. 你的名字是 {name}.")
])
nested_prompt_template2 = ChatPromptTemplate.from_messages([
    ("human", "我的问题是:{question}"),
])

prompt_template = ChatPromptTemplate.from_messages([
    nested_prompt_template1,
    nested_prompt_template2,
])

prompt = prompt_template.format_messages(name="小谷AI",question="如何学习人工智能?")
print(prompt)
print(type(prompt))

print(prompt[0].content)  #第一条消息
print(prompt[1].content)  #第二条消息



[SystemMessage(content='你是一个AI助手. 你的名字是 小谷AI.', additional_kwargs={}, response_metadata={}), HumanMessage(content='我的问题是:如何学习人工智能?', additional_kwargs={}, response_metadata={})]
<class 'list'>
你是一个AI助手. 你的名字是 小谷AI.
我的问题是:如何学习人工智能?


举例6 消息提示词模板

In [2]:
# 导入聊天消息类模板
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate,SystemMessagePromptTemplate

# 创建消息模板
system_template = "你是一个专家{role}"
system_message_prompt = SystemMessagePromptTemplate.from_template(system_template)

human_template = "给我解释{concept}，用浅显易懂的语言"
human_message_prompt = HumanMessagePromptTemplate.from_template(human_template)

# 组合成聊天提示模板
chat_prompt = ChatPromptTemplate.from_messages([system_message_prompt,
human_message_prompt])

# 格式化提示
formatted_messages = chat_prompt.format_messages(
    role="物理学家",
    concept="相对论"
)
print(formatted_messages)

[SystemMessage(content='你是一个专家物理学家', additional_kwargs={}, response_metadata={}), HumanMessage(content='给我解释相对论，用浅显易懂的语言', additional_kwargs={}, response_metadata={})]


In [ ]:
#与举例4的区别
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage
#创建实例


chat_prompt_template = ChatPromptTemplate.from_messages([
    SystemMessage(content="你是一个AI助手，你的名字是： {name}."),
    HumanMessage(content="我的问题是： {question}")
])

response = chat_prompt_template.invoke({})  #invoke()方法传入字典，空字典也可以
# response = chat_prompt_template.format_prompt(
# name="小谷AI",question="如何学习人工智能?")  #format_prompt()方法传入变量值
print(response.to_messages())
print(response.to_string())
print(type(response))

[SystemMessage(content='你是一个AI助手，你的名字是： {name}.', additional_kwargs={}, response_metadata={}), HumanMessage(content='我的问题是： {question}', additional_kwargs={}, response_metadata={})]
System: 你是一个AI助手，你的名字是： {name}.
Human: 我的问题是： {question}
<class 'langchain_core.prompt_values.ChatPromptValue'>


### 4.2.5 结合LLM

In [4]:
# 提供大模型
import os
import dotenv
from langchain_openai import ChatOpenAI
dotenv.load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")

chat_model = ChatOpenAI(model="gpt-4o-mini")


# 通过Chat提示词模板，创建提示词实例
from langchain_core.prompts import ChatPromptTemplate

#创建实例
chat_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个AI助手. 你的名字是 {name}."),
    ("human", "我的问题是:{question}"),
])

response = chat_prompt_template.invoke(
    {"name":"小谷AI","question":"如何学习人工智能?"}
)

# 通过大模型调用提示词
llm_response = chat_model.invoke(response)
print(llm_response.content)

学习人工智能（AI）是一个丰富而激动人心的旅程。以下是一些步骤和资源，帮助你有效地学习人工智能：

### 1. **基础知识**
   - **数学**：人工智能涉及很多数学，特别是线性代数、概率论和微积分。可以参考一些数学教材或在线课程来提升相关知识。
   - **编程**：熟悉至少一种编程语言，Python是人工智能领域的主流语言，很多库和框架（如TensorFlow和PyTorch）都是用Python编写的。

### 2. **计算机科学基础**
   - 学习数据结构和算法，这对理解AI模型的效率非常重要。
   - 了解计算机系统和操作系统的基础知识。

### 3. **人工智能概论**
   - 参加一些在线课程，比如：
     - Coursera上的《机器学习》课程（Andrew Ng教授）
     - edX的人工智能导论
   - 阅读相关书籍，例如《人工智能：一种现代的方法》（Artificial Intelligence: A Modern Approach）和《深度学习》（Deep Learning）书籍。

### 4. **深入学习**
   - **机器学习**：深入学习机器学习的基本概念，包括监督学习、无监督学习和强化学习。
   - **深度学习**：理解神经网络，卷积神经网络（CNN）、递归神经网络（RNN）等模型。

### 5. **实践项目**
   - 通过Kaggle等平台参与数据科学和机器学习竞赛，提升实战能力。
   - 自己动手做一些项目，比如图像识别、自然语言处理等。

### 6. **掌握工具和框架**
   - 学习使用机器学习和深度学习框架，如Scikit-learn、TensorFlow、Keras和PyTorch。
   - 熟悉数据处理工具，如Pandas和NumPy。

### 7. **参与社区**
   - 加入AI和机器学习的相关社区和论坛，如Stack Overflow、Reddit或学习微信群。
   - 参与开源项目，积累经验并与他人交流。

### 8. **持续学习**
   - 人工智能领域发展迅速，持续关注最新的研究和技术进展。可以通过关注学术论文、博客、社交媒体等进行学习。

### 9. **进阶研究**
   - 如果有兴趣，可以考虑研究生学习，深入某个特定

### 4.2.6 插入消息列表：MessagesPlaceholder


使用场景：当ChatPromptTemplate中的消息类型和个数不确定的时候，我们可以使用MessagesPlaceholder

举例1 

In [6]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

chat_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个AI助手. 你的名字是 {name}."),   
    MessagesPlaceholder(variable_name="msgs"), # 占位符
])

chat_prompt = chat_prompt_template.invoke({
    "name":"小谷AI",
    "msgs": [HumanMessage(content="如何学习人工智能?")]
}
)
print(chat_prompt)

messages=[SystemMessage(content='你是一个AI助手. 你的名字是 小谷AI.', additional_kwargs={}, response_metadata={}), HumanMessage(content='如何学习人工智能?', additional_kwargs={}, response_metadata={})]


举例2

In [8]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage


chat_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个AI助手. 你的名字是 {name}."),   
    MessagesPlaceholder(variable_name="msgs"), # 占位符
])

chat_prompt = chat_prompt_template.invoke({
    "name":"小谷AI",
    "msgs": [HumanMessage(content="如何学习人工智能?"),
             AIMessage(content="你可以从基础的数学和编程开始学习，然后逐步深入了解机器学习和深度学习等领域。")]
}
)
print(chat_prompt)

messages=[SystemMessage(content='你是一个AI助手. 你的名字是 小谷AI.', additional_kwargs={}, response_metadata={}), HumanMessage(content='如何学习人工智能?', additional_kwargs={}, response_metadata={}), AIMessage(content='你可以从基础的数学和编程开始学习，然后逐步深入了解机器学习和深度学习等领域。', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


举例3：存储对话历史内容

In [14]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一个AI助手. 你的名字是小智."),   
        MessagesPlaceholder(variable_name="chat_history"), # 占位符
        ("human", "{question}")
    ]
)

prompt_value = prompt.format_messages(
    chat_history=[
        HumanMessage(content="如何学习人工智能?"),
        AIMessage(content="你可以从基础的数学和编程开始学习，然后逐步深入了解机器学习和深度学习等领域。")
    ],
    question="我刚才的问题是什么？"
)

print(prompt_value)

[SystemMessage(content='你是一个AI助手. 你的名字是小智.', additional_kwargs={}, response_metadata={}), HumanMessage(content='如何学习人工智能?', additional_kwargs={}, response_metadata={}), AIMessage(content='你可以从基础的数学和编程开始学习，然后逐步深入了解机器学习和深度学习等领域。', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='我刚才的问题是什么？', additional_kwargs={}, response_metadata={})]


In [16]:
from langchain_openai import ChatOpenAI
import os
import dotenv
dotenv.load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")

chat_model = ChatOpenAI(model="gpt-4o-mini",)

chat_model.invoke(prompt_value).content

'你刚才的问题是“如何学习人工智能？”'

## 4.5 具体使用：少量样本示例的提示词模板

### 4.5.1 使用说明

FewShotPromptTemplate：与PromptTemplate一起使用

FewShotChatMessagePromptTemplate：与ChatPromptTemplate一起使用

Example solectors（示例选择器）：

### 4.5.2 FewShotPromptTemplate使用

举例1 未未提供示例的情况

In [ ]:
import os
import dotenv
from langchain_openai import ChatOpenAI
dotenv.load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

chat_model = ChatOpenAI(model="gpt-4o-mini",
                        temperature=0.4)

res = chat_model.invoke("2 🦜 9是多少?")
print(res.content)

2 🦜 9 的意思不太明确，可能是某种符号或表达方式。如果你能提供更多的上下文或解释这个符号的含义，我会更好地帮助你。


举例2 使用FewShotPromptTemplate

In [1]:
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate
# 创建PromptTemplate实例
example_prompt = PromptTemplate.from_template(
    template="input: {input}\noutput: {output}"
)

# 提供一系列示例
examples = [
    {"input": "北京今天天气怎么样", "output": "北京市"},
    {"input": "南京下雨么", "output": "南京市"},
    {"input": "武汉热吗", "output": "武汉市"},
]


# 创建FewShotPromptTemplate实例
few_shot_template = FewShotPromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
    suffix="input: {input}\noutput: ",
    input_variables=["input"]
)

few_shot_template.invoke({"input": "上海今天气温多少"})

StringPromptValue(text='input: 北京今天天气怎么样\noutput: 北京市\n\ninput: 南京下雨么\noutput: 南京市\n\ninput: 武汉热吗\noutput: 武汉市\n\ninput: 上海今天气温多少\noutput: ')

调用大模型以后：

In [3]:
import os
import dotenv   
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")      
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

chat_model = ChatOpenAI(model="gpt-4o-mini")



In [4]:
chat_model.invoke(few_shot_template.invoke({"input": "上海今天气温多少"})).content

'上海市'

举例3：

In [9]:
# 创建提示词模板
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

# 创建提示词模板，配置一个提示词模板，将一个示例格式化为字符串
prompt_template = "你是一个数学专家，算式：input: {input}，值：output: {output}，使用：{description}"

# 这是一个提示词模板，用于设置每个示例的格式
prompt_sample = PromptTemplate.from_template(prompt_template)

# 提供示例
examples = [
    {"input": "2 + 2", "output": "4", "description": "加法"},
    {"input": "3 * 5", "output": "15", "description": "乘法"},
    {"input": "10 - 4", "output": "6", "description": "减法"},
]

# 创建FewShotPromptTemplate实例
prompt = FewShotPromptTemplate(
    example_prompt=prompt_sample,
    examples=examples,
    suffix="你是一个数学专家，算式：input: {input}，值：output: ",
    input_variables=["input", "output"]
)

print(prompt.invoke({"input": "8 / 2", "output": ""}))


# 初始化大模型
import os
import dotenv   
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")  
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

chat_model = ChatOpenAI(model="gpt-4o-mini")
# 调用大模型
response = chat_model.invoke(prompt.invoke({"input": "8 / 2", "output": ""}))
print(response.content)

text='你是一个数学专家，算式：input: 2 + 2，值：output: 4，使用：加法\n\n你是一个数学专家，算式：input: 3 * 5，值：output: 15，使用：乘法\n\n你是一个数学专家，算式：input: 10 - 4，值：output: 6，使用：减法\n\n你是一个数学专家，算式：input: 8 / 2，值：output: '
4，使用：除法


### 4.5.3 FewshotChatMessagePromptTemplate的使用

举例1：

In [11]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate, ChatPromptTemplate

# 1、示例消息格式
examples =[
    {"input":"1+1等于几？", "output":"1+1等于2"},
    {"input":"法国的首都是？", "output":"巴黎"},
]
# 2、定义示例的消息格式提示词模板
msg_example_prompt = ChatPromptTemplate.from_messages([
    ("human","{input}"),
    ("ai","{output}"),
])

# 3、定义FewShotChatMessagePromptTemplate对象
few_shot_chat_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=msg_example_prompt,
    examples=examples,
)

# 4、输出格式化后的消息
print(few_shot_chat_prompt.invoke({}))

messages=[HumanMessage(content='1+1等于几？', additional_kwargs={}, response_metadata={}), AIMessage(content='1+1等于2', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='法国的首都是？', additional_kwargs={}, response_metadata={}), AIMessage(content='巴黎', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


举例2：
使用方式：将原始输入和被选中的示例组一起加入Chat提示词模板中

In [15]:
# 1、导入相关包
from langchain_core.prompts import (FewShotChatMessagePromptTemplate
                                    , ChatPromptTemplate)

# 2、定义示例组
examples =[
    {"input":"2🦜2", "output": "4"},
    {"input":"2🦜3", "output": "8"},
]

# 3、定义示例的消息格式提示词模板
example_prompt = ChatPromptTemplate.from_messages([
    ("human","计算：{input}"),
    ("ai","结果是：{output}"),
])

# 4、定义FewShotChatMessagePromptTemplate对象
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,  #示例提示词模板
    examples=examples,  #示例组
)

# 5、输出完整提示词的消息模板
final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一个数学专家。"),
        few_shot_prompt,  #嵌入FewShotChatMessagePromptTemplate对象
        ("human", "计算：{input}"),
    ]
)

# 6、提供大模型
import os
import dotenv
from langchain_openai import ChatOpenAI
dotenv.load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")

chat_model = ChatOpenAI(model="gpt-4o-mini",temperature=0.4)

# 7、调用大模型
response = chat_model.invoke(
    final_prompt.invoke({"input": "2 🦜4"})
).content
print(response)

结果是：16


### 4.5.4 Example selectors(示例选择器)

避免盲目传递所有示例，减少 token 消耗的同时，还可以提升输出效果。

语义相似选择：通过余弦相似度等度量方式评估语义相关性，选择与输入问题最相似的 k 个示例。

长度选择：根据输入文本的长度，从候选示例中筛选出长度最匹配的示例。增强模型对文本结构的理解。比语义相似度计算更轻量，适合对响应速度要求高的场景。

最大边际相关示例选择：优先选择与输入问题语义相似的示例；同时，通过惩罚机制避免返回同质化的内容

举例1：

In [ ]:
# 1.导入相关包
from langchain_community.vectorstores import Chroma
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
import os
import dotenv
from langchain_openai import OpenAIEmbeddings
dotenv.load_dotenv()
# 2.定义嵌入模型
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-ada-002"
)
# 3.定义示例组
examples = [
    {
        "question": "谁活得更久，穆罕默德·阿里还是艾伦·图灵?",
        "answer": """
        接下来还需要问什么问题吗？
        追问：穆罕默德·阿里去世时多大年纪？
        中间答案：穆罕默德·阿里去世时享年74岁。
        """,
     },
    {
        "question": "craigslist的创始人是什么时候出生的？",
        "answer": """
        接下来还需要问什么问题吗？
        追问：谁是craigslist的创始人？
        中级答案：Craigslist是由克雷格·纽马克创立的。
        """,
    },
    {
        "question": "谁是乔治·华盛顿的外祖父？",
        "answer": """
        接下来还需要问什么问题吗？
        追问：谁是乔治·华盛顿的母亲？
        中间答案：乔治·华盛顿的母亲是玛丽·鲍尔·华盛顿。
        """,
    },
    {
        "question": "《大白鲨》和《皇家赌场》的导演都来自同一个国家吗？",
        "answer": """
        接下来还需要问什么问题吗？
        追问：《大白鲨》的导演是谁？
        中级答案：《大白鲨》的导演是史蒂文·斯皮尔伯格。
        """,
    },
]
# 4.定义示例选择器
example_selector = SemanticSimilarityExampleSelector.from_examples(
    # 这是可供选择的示例列表
    examples,
    # 这是用于生成嵌入的嵌入类，用于衡量语义相似性
    embeddings_model,
    # 这是用于存储嵌入并进行相似性搜索的 VectorStore 类
    Chroma,
    # 这是要生成的示例数量
    k=1,
)
# 选择与输入最相似的示例

question = "玛丽·鲍尔·华盛顿的父亲是谁?"
selected_examples = example_selector.select_examples({"question": question})
print(f"与输入最相似的示例：{selected_examples}")

for example in selected_examples:
    print("\n")
for k, v in example.items():
    print(f"{k}: {v}")

与输入最相似的示例：[{'answer': '\n        接下来还需要问什么问题吗？\n        追问：谁是乔治·华盛顿的母亲？\n        中间答案：乔治·华盛顿的母亲是玛丽·鲍尔·华盛顿。\n        ', 'question': '谁是乔治·华盛顿的外祖父？'}]


answer: 
        接下来还需要问什么问题吗？
        追问：谁是乔治·华盛顿的母亲？
        中间答案：乔治·华盛顿的母亲是玛丽·鲍尔·华盛顿。
        
question: 谁是乔治·华盛顿的外祖父？


举例2：结合FewShotPromptTemplate使用
这里使用FAISS，需要安装，pip install faiss-cpu

In [12]:
#1.导入相关包
from langchain_community.vectorstores import FAISS
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
from langchain_openai import OpenAIEmbeddings
# 2.定义示例提示词模版
example_prompt = PromptTemplate.from_template(
    template="Input: {input}\nOutput: {output}",
)
# 3.创建一个示例提示词模版
examples = [
    {"input": "高兴", "output": "悲伤"},
    {"input": "高", "output": "矮"},
    {"input": "长", "output": "短"},
    {"input": "精力充沛", "output": "无精打采"},
    {"input": "阳光", "output": "阴暗"},
    {"input": "粗糙", "output": "光滑"},
    {"input": "干燥", "output": "潮湿"},
    {"input": "富裕", "output": "贫穷"},
]
# 4.定义嵌入模型
embeddings = OpenAIEmbeddings(
    model="text-embedding-ada-002"
)
# 5.创建语义相似性示例选择器
example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples,
    embeddings,
    FAISS,
    k=2,
)
#或者
#example_selector = SemanticSimilarityExampleSelector(
# examples,
# embeddings,
# FAISS,
# k=2
#)


# 6.定义小样本提示词模版
similar_prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    prefix="给出每个词组的反义词",
    suffix="Input: {word}\nOutput:",
    input_variables=["word"],
)
response = similar_prompt.invoke({"word":"忧郁"})
print(response.text)


给出每个词组的反义词

Input: 高兴
Output: 悲伤

Input: 阳光
Output: 阴暗

Input: 忧郁
Output:


In [6]:
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"]=os.getenv("OPENAI_BASE_URL")

llm = ChatOpenAI(model="gpt-4o-mini")

res = llm.invoke(response)
print(res.content)


快乐


## 4.6 具体使用：PipelinePromptTemplate(了解)


用于将多个提示模板按顺序组合成处理管道，实现分阶段、模块化的提示构建。它的核心作用类似于软件开发中的管道模式（Pipeline Pattern），通过串联多个提示处理步骤，实现复杂的提示生成逻辑。

特点：
将复杂提示拆解为多个处理阶段，每个阶段使用独立的提示模板前一个模板的输出作为下一个模板的输入变量

使用场景：解决单一超大提示模板难以维护的问题

说明：PipelinePromptTemplate在langchain 0.3.22版本中被标记为过时，在langchain-core==1.0之前不会删除它。

In [9]:
from langchain_core.prompts.prompt import PromptTemplate

# 阶段1：问题分析
analysis_template = PromptTemplate.from_template("""
    分析这个问题：{question}
    关键要素：
    """)

# 阶段2：知识检索
retrieval_template = PromptTemplate.from_template("""
    基于以下要素搜索资料：
    {analysis_result}
    搜索关键词：
    """)

# 阶段3：生成最终回答
answer_template = PromptTemplate.from_template("""
    综合以下信息回答问题：
    {retrieval_result}
    最终答案：
    """)

# 逐步执行管道提示
pipeline_prompts = [
    ("analysis_result", analysis_template),
    ("retrieval_result", retrieval_template)
]


my_input = {"question": "量子计算的优势是什么？"}

print(pipeline_prompts)
# [('analysis_result', PromptTemplate(input_variables=['question'], input_types={},
# partial_variables={}, template='\n分析这个问题：{question}\n关键要素：\n')), ('retrieval_result',
# PromptTemplate(input_variables=['analysis_result'], input_types={}, partial_variables={},
# template='\n基于以下要素搜索资料：\n{analysis_result}\n搜索关键词：\n'))]


for name, prompt in pipeline_prompts:
    # 调用当前提示模板并获取字符串结果
    result = prompt.invoke(my_input).to_string()
    # 将结果添加到输入字典中供下一步使用
    my_input[name] = result


# 生成最终答案
my_output = answer_template.invoke(my_input).to_string()
print(my_output)

[('analysis_result', PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='\n    分析这个问题：{question}\n    关键要素：\n    ')), ('retrieval_result', PromptTemplate(input_variables=['analysis_result'], input_types={}, partial_variables={}, template='\n    基于以下要素搜索资料：\n    {analysis_result}\n    搜索关键词：\n    '))]

    综合以下信息回答问题：
    
    基于以下要素搜索资料：
    
    分析这个问题：量子计算的优势是什么？
    关键要素：
    
    搜索关键词：
    
    最终答案：
    


## 4.7 具体使用：自定义提示词模版(了解)

在创建prompt时，我们也可以按照自己的需求去创建自定义的提示模版。
步骤：
自定义类继承提示词基类模版BasePromptTemplate
重写format、format_prompt、from_template方法

举例

In [ ]:
# 1.导入相关包
from typing import List, Dict, Any
from langchain_core.prompts import BasePromptTemplate
from langchain_core.prompts import PromptTemplate
from langchain_core.prompt_values import PromptValue

# 2.自定义提示词模版
class SimpleCustomPrompt(BasePromptTemplate):
    """简单自定义提示词模板"""
    template: str # type: ignore

def __init__(self, template: str, **kwargs): # type: ignore
    # 使用PromptTemplate解析输入变量
    # prompt = PromptTemplate.from_template(template)
    # 
    # 
    super().__init__(
        input_variables=prompt.input_variables,
        template=template,
        **kwargs
    )

def format(self, **kwargs: Any) -> str: # type: ignore
    """格式化提示词"""
    # print("kwargs:", kwargs)
    # print("self.template:", self.template)

    return self.template.format(**kwargs)

def format_prompt(self, **kwargs: Any) -> PromptValue:
    """实现抽象方法"""
    return PromptValue(text=self.format(**kwargs))

@classmethod
def from_template(cls, template: str, **kwargs) -> "SimpleCustomPrompt": # type: ignore
    """从模板创建实例"""
    return cls(template=template, **kwargs)

# 3.使用自定义提示词模版
custom_prompt = PromptTemplate.from_template(
    template="请回答关于{subject}的问题：{question}"
)

# 4.格式化提示词
formatted = custom_prompt.format(
    subject="人工智能",
    question="什么是LLM？"
)

print(formatted)

请回答关于人工智能的问题：什么是LLM？


## 4.8 从文档中加载Prompt(了解)

一方面，将想要设定prompt所支持的格式保存为JSON或者YAML格式文件。

另一方面，通过读取指定路径的格式化文件，获取相应的prompt。

目的与使用场景：

为了便于共享、存储和加强对prompt的版本控制。

当我们的prompt模板数据较大时，我们可以使用外部导入的方式进行管理和维护。

### 4.8.1 yaml格式提示词

asset下创建yaml文件：prompt.yaml

yaml文件内写入：

_type:
  "prompt"
input_variables:
  ["name","what"]
template:
  "请给{name}讲一个关于{what}的故事"

In [25]:
from langchain_core.prompts import load_prompt
from dotenv import load_dotenv

load_dotenv()

prompt = load_prompt("asset/prompt.yaml", encoding="utf-8")

print(prompt)

print(prompt.format(name="年轻人", what="滑稽"))

input_variables=['name', 'what'] input_types={} partial_variables={} template='请给{name}讲一个关于{what}的故事'
请给年轻人讲一个关于滑稽的故事


### 4.8.2 json格式提示词

Jason文件中写入：

{
    "_type": "prompt",
    "input_variables": ["name", "what"],
    "template": "请{name}讲一个{what}的故事。"
}

In [27]:
from langchain_core.prompts import load_prompt
from dotenv import load_dotenv

load_dotenv()

prompt = load_prompt("asset/prompt.json",encoding="utf-8")
print(prompt.format(name="张三",what="搞笑的"))

请张三讲一个搞笑的的故事。


# 5. Model I/O之Output Parsers

语言模型返回的内容通常都是字符串的格式（文本格式），但在实际AI应用开发过程中，往往希望model可以返回更直观、更格式化的内容，以确保应用能够顺利进行后续的逻辑处理。此时，LangChain提供的输出解析器就派上用场了。


输出解析器（Output Parser）负责获取 LLM 的输出并将其转换为更合适的格式。这在应用开发中及其重要。

## 5.1 输出解析器的分类

LangChain有许多不同类型的输出解析器

StrOutputParser ：字符串解析器

JsonOutputParser ：JSON解析器，确保输出符合特定JSON对象格式

XMLOutputParser ：XML解析器，允许以流行的XML格式从LLM获取结果

CommaSeparatedListOutputParser ：CSV解析器，模型的输出以逗号分隔，以列表形式返回输出

DatetimeOutputParser ：日期时间解析器，可用于将 LLM 输出解析为日期时间格式

## 5.2 具体解析器的使用

### 1.字符串解析器 StrOutputParser

StrOutputParser 简单地将任何输入转换为字符串。它是一个简单的解析器，从结果中提取content字段

举例：将一个对话模型的输出结果，解析为字符串输出

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

chat_model = ChatOpenAI(model="gpt-4o-mini")

messages = [
    SystemMessage(content="将以下内容从英语翻译成中文"),
    HumanMessage(content="It's a nice day today"),
]

result = chat_model.invoke(messages)
print(type(result))
print(result)


parser = StrOutputParser()
#使用parser处理model返回的结果
response = parser.invoke(result)
print(type(response))
print(response)

<class 'langchain_core.messages.ai.AIMessage'>
content='今天是个好日子。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 25, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f97eff32c5', 'id': 'chatcmpl-D4l4zIcKB9jhQSsodvEyARg1bE9qo', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019c1dcf-64fa-79c3-a542-2860c6e4c5ee-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 25, 'output_tokens': 8, 'total_tokens': 33, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
<class 'str'>
今天是个好日子。


### 2.JSON解析器 JsonOutputParser

JsonOutputParser，即JSON输出解析器，是一种用于将大模型的自由文本输出转换为结构化JSON数据的工具。

适合场景：特别适用于需要严格结构化输出的场景，比如 API 调用、数据存储或下游任务处理。

实现方式

方式1：用户自己通过提示词指明返回Json格式

方式2：借助JsonOutputParser的get_format_instructions() ，生成格式说明，指导模型输出JSON 结构

举例1：

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate

chat_model = ChatOpenAI(model="gpt-4o-mini")

chat_prompt_template = ChatPromptTemplate.from_messages([
    ("system","你是一个靠谱的{role}"),
    ("human","{question}")
])

parser = JsonOutputParser()

# 方式1：
result = chat_model.invoke(chat_prompt_template.format_messages(
    role="人工智能专家",
    question="人工智能用英文怎么说？问题用q表示，答案用a表示，返回一个JSON格式"))#提示词指明返回Json
print(result)
print(type(result))

parser.invoke(result)

# 方式2：
# 2_chains = chat_prompt_template | chat_model | parser
# 2_chains.invoke({"role":"人工智能专家","question" : "人工智能用英文怎么说？问题用q表示，答案用a表示，返回一个JSON格式"})

content='```json\n{\n  "q": "人工智能怎么用英文说？",\n  "a": "Artificial Intelligence"\n}\n```' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 40, 'total_tokens': 68, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f97eff32c5', 'id': 'chatcmpl-D4l8UFuVKEaWBRBbSN43FFWCenmkl', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019c1dd2-b38a-7c92-bcb8-9ddf89923521-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 40, 'output_tokens': 28, 'total_tokens': 68, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
<class 'langchain_core.messages.ai.AIMessage'>


{'q': '人工智能怎么用英文说？', 'a': 'Artificial Intelligence'}

举例2：使用指定的JSON格式

In [1]:
from langchain_core.output_parsers import JsonOutputParser

output_parser = JsonOutputParser()
# 返回一些指令或模板，这些指令告诉系统如何解析或格式化输出数据
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

Return a JSON object.


基于此：

In [ ]:
# 引入依赖包
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# 初始化语言模型
chat_model = ChatOpenAI(model="gpt-4o-mini")

joke_query = "告诉我一个笑话。"

# 定义Json解析器
parser = JsonOutputParser()

# 定义提示词模版
# 注意，提示词模板中需要部分格式化解析器的格式要求format_instructions
prompt = PromptTemplate(
    template="回答用户的查询.\n{format_instructions}\n{query}\n",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

# 使用LCEL语法组合一个简单的链
chain = prompt | chat_model | parser

# 执行链
output = chain.invoke({"query": "给我讲一个笑话"})
print(output)

{'joke': '为什么数学书总是很忧伤？因为它有太多的问题！'}


### 3.XML解析器XML OutputParser

XMLOutputParser，将模型的自由文本输出转换为可编程处理的 XML 数据。

如何实现：在 PromptTemplate 中指定 XML 格式要求，让模型返回 <tag>content</tag> 形式的数据。

注意：XMLOutputParser 不会直接将模型的输出保持为原始XML字符串，而是会解析XML并转换成Python字典（或类似结构化的数据）。目的是为了方便程序后续处理数据，而不是单纯保留XML格式。

举例1：不使用XMLOutputParser，通过大模型的能力，返回xml格式数据

In [4]:
# 初始化语言模型
chat_model = ChatOpenAI(model="gpt-4o-mini")

# 测试模型的xml解析效果
actor_query = "生成汤姆·汉克斯的简短电影记录"
output = chat_model.invoke(f"""{actor_query}请将影片附在<movie></movie>标签中"""
)

print(type(output)) # <class 'langchain_core.messages.ai.AIMessage'>
print(output.content)

<class 'langchain_core.messages.ai.AIMessage'>
汤姆·汉克斯是一位备受赞誉的美国演员和制片人，以下是他的一些经典电影记录：

<movie>
  <title>福雷斯特·甘普</title>
  <year>1994</year>
  <description>一个智力平平但心地善良的男人，福雷斯特走过了美国历史上的重要时刻，经历了无数冒险，并赢得了爱情。</description>
</movie>

<movie>
  <title>拯救大兵瑞恩</title>
  <year>1998</year>
  <description>在第二次世界大战期间，一组士兵被派往敌后，寻找并救出一名失踪的士兵，体现了友谊和奉献的主题。</description>
</movie>

<movie>
  <title>等级清理</title>
  <year>2002</year>
  <description>根据真实故事改编，讲述了一名空乘人员在一场空难中努力求生的经历，展现了人性的韧性。</description>
</movie>

<movie>
  <title>贫民窟的百万富翁</title>
  <year>2008</year>
  <description>故事围绕一名孤儿在印度贫民区成长并参加电视问答节目，他的经历揭示了命运与希望。</description>
</movie>

<movie>
  <title>卡拉韦尔</title>
  <year>2012</year>
  <description>讲述了大导演对创作和生活的反思，展现了艺术的力量与真实的情感。</description>
</movie>

这些影片展示了汉克斯作为一名演员的广泛才能和影响力。


举例2：体会XMLOutputParser的格式

In [5]:
from langchain_core.output_parsers import XMLOutputParser

output_parser = XMLOutputParser()
# 返回一些指令或模板，这些指令告诉系统如何解析或格式化输出数据
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

The output should be formatted as a XML file.
1. Output should conform to the tags below.
2. If tags are not given, make them on your own.
3. Remember to always open and close all the tags.

As an example, for the tags ["foo", "bar", "baz"]:
1. String "<foo>
   <bar>
      <baz></baz>
   </bar>
</foo>" is a well-formatted instance of the schema.
2. String "<foo>
   <bar>
   </foo>" is a badly-formatted instance.
3. String "<foo>
   <tag>
   </tag>
</foo>" is a badly-formatted instance.

Here are the output tags:
```
None
```


举例3：XMLOutputParser 的使用

In [10]:
# 1.导入相关包
from langchain_core.output_parsers import XMLOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# 2. 初始化语言模型
chat_model = ChatOpenAI(model="gpt-4o-mini")

# 3.测试模型的xml解析效果
actor_query = "生成汤姆·汉克斯的简短电影记录,使用中文回复"

# 4.定义XMLOutputParser对象
parser = XMLOutputParser()

# 5.定义提示词模版对象
# prompt = PromptTemplate(
# template="{query}\n{format_instructions}",
# input_variables=["query","format_instructions"],
# partial_variables={"format_instructions": parser.get_format_instructions()},
#)

prompt_template = PromptTemplate.from_template(
    template = "用户的问题：{query}\n使用的格式：{format_instructions}"
    )

prompt_template1 = prompt_template.partial(format_instructions=parser.get_format_instructions())

response = chat_model.invoke(prompt_template1.format(query=actor_query))
print(response.content)

```xml
<电影记录>
    <演员>
        <姓名>汤姆·汉克斯</姓名>
        <国籍>美国</国籍>
        <职业>演员、制片人、导演</职业>
    </演员>
    <获奖情况>
        <奖项>
            <名称>奥斯卡最佳男主角</名称>
            <年份>1994</年份>
            <电影>费城故事</电影>
        </奖项>
        <奖项>
            <名称>奥斯卡最佳男主角</名称>
            <年份>1995</年份>
            <电影>阿甘正传</电影>
        </奖项>
    </获奖情况>
    <代表作品>
        <电影>
            <标题>阿甘正传</标题>
            <年份>1994</年份>
        </电影>
        <电影>
            <标题>拯救大兵瑞恩</标题>
            <年份>1998</年份>
        </电影>
        <电影>
            <标题>费城</标题>
            <年份>1993</年份>
        </电影>
    </代表作品>
</电影记录>
```


继续：用parser不会输出为xml，类似与json

In [7]:
# 方式1
response = chat_model.invoke(prompt_template1.format(query=actor_query))
result = parser.invoke(response)
print(result)
print(type(result))


# 方式2
# 2_chains = prompt_template1 | chat_model | parser
# result = 2_chains.invoke({"query":actor_query})
# print(result)
# print(type(result))

{'汤姆汉克斯电影记录': [{'电影': [{'标题': '拯救大兵瑞恩'}, {'年份': '1998'}, {'角色': '米勒中尉'}, {'类型': '战争/剧情'}]}, {'电影': [{'标题': '阿甘正传'}, {'年份': '1994'}, {'角色': '福ores特·甘'}, {'类型': '剧情/爱情'}]}, {'电影': [{'标题': '飞越未来'}, {'年份': '2004'}, {'角色': '沃尔特·米特尼'}, {'类型': '冒险/科幻'}]}, {'电影': [{'标题': '极度空间'}, {'年份': '1993'}, {'角色': '吉姆·阿普尔顿'}, {'类型': '惊悚/剧情'}]}, {'电影': [{'标题': '大白鲨'}, {'年份': '1975'}, {'角色': '马丁·布罗迪'}, {'类型': '惊悚/恐怖'}]}]}
<class 'dict'>


举例4：与前例类似

In [8]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import XMLOutputParser

model = ChatOpenAI(model="gpt-4o-mini")

actor_query = "生成周星驰的简化电影作品列表，按照最新的时间降序，必要时使用中文"
# 设置解析器 + 将指令注入提示模板。
parser = XMLOutputParser()
prompt = PromptTemplate(
    template="回答用户的查询。\n{format_instructions}\n{query}\n",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

chain = prompt | model | parser
output = chain.invoke({"query": actor_query})
print(output)

{'周星驰作品': [{'电影': [{'标题': '美人鱼'}, {'年份': '2016'}]}, {'电影': [{'标题': '西游降魔篇'}, {'年份': '2013'}]}, {'电影': [{'标题': '长江七号'}, {'年份': '2008'}]}, {'电影': [{'标题': '卧虎藏龙'}, {'年份': '2000'}]}, {'电影': [{'标题': '食神'}, {'年份': '1996'}]}, {'电影': [{'标题': '大话西游'}, {'年份': '1995'}]}]}


### 4.列表解析器CommaSeparatedListOutputParser

列表解析器：利用此解析器可以将模型的文本响应转换为一个用逗号分隔的列表（List[str]） 。

举例1

In [11]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

output_parser = CommaSeparatedListOutputParser()

# 返回一些指令或模板，这些指令告诉系统如何解析或格式化输出数据
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

messages = "大象,猩猩,狮子"
result = output_parser.parse(messages)
print(result)
print(type(result))

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`
['大象', '猩猩', '狮子']
<class 'list'>


举例2

In [1]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import CommaSeparatedListOutputParser
import os
import dotenv   
dotenv.load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
# 初始化语言模型
chat_model = ChatOpenAI(model="gpt-4o-mini")

# 创建解析器
output_parser = CommaSeparatedListOutputParser()

# 创建LangChain提示模板
chat_prompt = PromptTemplate.from_template(
    "生成5个关于{text}的列表.\n\n{format_instructions}",
    partial_variables={
        "format_instructions": output_parser.get_format_instructions()
})

# 提示模板与输出解析器传递输出
# chat_prompt = chat_prompt.partial(format_instructions=output_parser.get_format_instructions())

# 将提示和模型合并以进行调用
chain = chat_prompt | chat_model | output_parser
res = chain.invoke({"text": "电影"})
print(res)
print(type(res))#<class 'list'>

APIConnectionError: Connection error.

举例3：

In [3]:
from langchain_core.prompts import HumanMessagePromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import CommaSeparatedListOutputParser

# 初始化语言模型
chat_model = ChatOpenAI(model="gpt-4o-mini")

output_parser = CommaSeparatedListOutputParser()

chat_prompt = ChatPromptTemplate.from_messages([
   # ("human", "{request}\n{format_instructions}")
   HumanMessagePromptTemplate.from_template("{request}\n{format_instructions}"),
])


chain = chat_prompt | chat_model | output_parser
resp = chain.invoke({"request": "给我5个心情", "format_instructions":
                     output_parser.get_format_instructions()})
print(resp)
print(type(resp))

['快乐', '悲伤', '平静', '焦虑', '兴奋']
<class 'list'>


方式2

In [4]:
model_request = chat_prompt.format_messages(
    request = "给我5个心情",
    format_instructions = output_parser.get_format_instructions()
)

result = chat_model.invoke(model_request)
resp = output_parser.parse(result.content)
print(resp)
print(type(resp))

['快乐', '悲伤', '兴奋', '焦虑', '平静']
<class 'list'>


方式3

In [27]:
result = chat_model.invoke(model_request)
resp = output_parser.invoke(result)
print(resp)
print(type(resp))

['快乐', '悲伤', '焦虑', '兴奋', '平静']
<class 'list'>


### 5.日期解析器 DatetimeOutputParser (了解)新版已经没有了

利用此解析器可以直接将LLM输出解析为日期时间格式。
get_format_instructions()： 获取日期解析的格式化指令，

指令为："Write a datetime string
that matches the following pattern: '%Y-%m-%dT%H:%M:%S.%fZ'。

举例：1206-08-16T17:39:06.176399Z

举例1：

In [ ]:
from langchain_core.output_parsers import BaseOutputParser

output_parser = BaseOutputParser

format_instructions = output_parser.get_format_instructions()
print(format_instructions)

ImportError: cannot import name 'DatetimeOutputParser' from 'langchain_core.output_parsers' (c:\Users\huan.zheng\AppData\Local\miniconda3\envs\langchain1\Lib\site-packages\langchain_core\output_parsers\__init__.py)

举例2：

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import HumanMessagePromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.output_parsers import DatetimeOutputParser

chat_model = ChatOpenAI(model="gpt-4o-mini")

chat_prompt = ChatPromptTemplate.from_messages([
    ("system","{format_instructions}"),
    ("human", "{request}")
])

output_parser = DatetimeOutputParser()



ImportError: cannot import name 'DatetimeOutputParser' from 'langchain_community.output_parsers' (c:\Users\huan.zheng\AppData\Local\miniconda3\envs\langchain1\Lib\site-packages\langchain_community\output_parsers\__init__.py)

In [18]:
# 验证模块是否存在及对应路径
try:
    # 先尝试最新版路径
    from langchain_community.output_parsers.datetime import DatetimeOutputParser
    print("✅ 模块存在，导入成功（最新版路径）")
except ImportError as e1:
    try:
        # 再尝试过渡版路径
        from langchain_community.output_parsers import DatetimeOutputParser
        print("✅ 模块存在，导入成功（过渡版路径）")
    except ImportError as e2:
        print("❌ 你的环境中无 DatetimeOutputParser 模块：")
        print("   原因1：langchain_community 版本过低（＜0.1.0）")
        print("   原因2：未安装 langchain_community 包")

❌ 你的环境中无 DatetimeOutputParser 模块：
   原因1：langchain_community 版本过低（＜0.1.0）
   原因2：未安装 langchain_community 包


方式1

In [11]:
model_request = chat_prompt.format_messages(
    request="中华人民共和国是什么时候成立的",
    format_instructions=output_parser.get_format_instructions()
)

response = chat_model.invoke(model_request)
result = output_parser.invoke(response)
print(result)
print(type(result))

['1949年10月1日']
<class 'list'>


方式2

In [20]:
chain = chat_prompt | chat_model | output_parser
resp = chain.invoke({"request":"中华人民共和国是什么时候成立的",
                     "format_instructions":output_parser.get_format_instructions()})
print(resp)
print(type(resp))

1949-10-01 00:00:00
<class 'datetime.datetime'>
